<a href="https://colab.research.google.com/github/debo-ogunnowo/Prompt-Inference-System/blob/main/p_reconstructor_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
import torch

In [5]:
df = pd.read_csv('prompt_repsonse_clean.csv')
df.dropna(how='all', inplace=True)

In [6]:
df.head()

,message,cleaned_response
0,I think I'm mixing metaphors in this paragraph...,"You're close, but mixing metaphors can be tric..."
1,I keep restating my research question in almos...,You can condense the discussion to reduce redu...
2,Can you weave a reference to Table 2 into this...,"Here's a revised paragraph: ""As observed in th..."
3,"This reminder email buries the actual ask, can...",Here's a rewritten version with a clearer call...
4,These items in the sentence don't have paralle...,You're correct that the sentence has non-paral...


In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df.to_csv('train.csv', index=False)
test_df.to_csv('test.csv', index=False)

print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

Training set size: 912
Test set size: 229


In [8]:
RESPONSE_COL = 'cleaned_response'
PROMPT_COL = 'message'

In [9]:
# TF-IDF Retreival Baseline

print("\nBuilding TF-IDF retrieval baseline...")

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True
)

# Fit on training responses
train_response_vectors = tfidf.fit_transform(train_df[RESPONSE_COL].values)

# Transform test responses using the same fitted vectorizer
test_response_vectors = tfidf.transform(test_df[RESPONSE_COL].values)

# For each test response, find the most similar training response
# and return its corresponding original prompt
similarity_matrix = cosine_similarity(test_response_vectors, train_response_vectors)
nearest_indices = similarity_matrix.argmax(axis=1)

baseline_reconstructions = train_df[PROMPT_COL].values[nearest_indices]
test_df['baseline_reconstruction'] = baseline_reconstructions

print("TF-IDF retrieval complete.")


Building TF-IDF retrieval baseline...
TF-IDF retrieval complete.


In [10]:
# Cosine Similarity Evaluation
print("\nEvaluating baseline with sentence embedding cosine similarity...")

sem_model = SentenceTransformer('all-MiniLM-L6-v2')

original_embeddings = sem_model.encode(
    test_df[PROMPT_COL].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

baseline_embeddings = sem_model.encode(
    test_df['baseline_reconstruction'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

similarity_scores = util.cos_sim(
    original_embeddings,
    baseline_embeddings
).diagonal().cpu().numpy()

test_df['baseline_similarity'] = similarity_scores


Evaluating baseline with sentence embedding cosine similarity...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [15]:
# Results
print("TF-IDF RETRIEVAL BASELINE RESULTS\n")
print(f"Mean cosine similarity:       {similarity_scores.mean():.4f}")
print(f"Median cosine similarity:     {np.median(similarity_scores):.4f}")
print(f"Standard deviation:           {similarity_scores.std():.4f}")
print(f"Minimum:                      {similarity_scores.min():.4f}")
print(f"Maximum:                      {similarity_scores.max():.4f}")
print(f"Proportion above 0.9:         {(similarity_scores > 0.9).mean():.4f}")
print(f"Proportion above 0.7:         {(similarity_scores > 0.7).mean():.4f}")
print(f"Proportion above 0.5:         {(similarity_scores > 0.5).mean():.4f}")

TF-IDF RETRIEVAL BASELINE RESULTS

Mean cosine similarity:       0.3831
Median cosine similarity:     0.3396
Standard deviation:           0.2113
Minimum:                      0.0164
Maximum:                      0.9840
Proportion above 0.9:         0.0218
Proportion above 0.7:         0.0873
Proportion above 0.5:         0.2707


In [17]:
# P-Reconstructor Comparison
print("RECONSTRUCTION MODEL COMPARISON SUMMARY\n")
print(f"{'Model':<35} {'Mean Sim':>9} {'Above 0.7':>10} {'Above 0.5':>10}")
print("-"*65)
print(f"{'TF-IDF Retrieval Baseline':<35} {similarity_scores.mean():>9.4f} {(similarity_scores > 0.7).mean():>10.4f} {(similarity_scores > 0.5).mean():>10.4f}")
print(f"{'P-Reconstructor (Phi-3-Mini-4K-Instruct)':<35} {'0.8296':>9} {'0.8472':>10} {'0.9563':>10}")

RECONSTRUCTION MODEL COMPARISON SUMMARY

Model                                Mean Sim  Above 0.7  Above 0.5
-----------------------------------------------------------------
TF-IDF Retrieval Baseline              0.3831     0.0873     0.2707
P-Reconstructor (Phi-3-Mini-4K-Instruct)    0.8296     0.8472     0.9563


In [13]:
# Sample Qualitative examples
print("\nSample Baseline Reconstructions:")
for i in range(min(5, len(test_df))):
    print(f"\nOriginal:  {test_df[PROMPT_COL].iloc[i][:100]}")
    print(f"Baseline:  {test_df['baseline_reconstruction'].iloc[i][:100]}")
    print(f"Similarity: {similarity_scores[i]:.4f}")


Sample Baseline Reconstructions:

Original:  What do you know about Korsakoff's syndrome
Baseline:  elaborate on the diagnosis and treatment of acute stress disorder
Similarity: 0.2413

Original:  search engine indexing, how does that work behind the scenes
Baseline:  database indexes, what do they do and how do they speed things up
Similarity: 0.7093

Original:  Compare the strengths and weaknesses of parliamentary and presidential systems of government
Baseline:  What is the importance of separation of powers in a political system?
Similarity: 0.5172

Original:  how do you classify different types of energy sources
Baseline:  break down renewable vs non renewable energy for me
Similarity: 0.4419

Original:  How do mangrove trees survive in saline coastal environments where most plants cannot
Baseline:  darwin's natural selection, what's the core idea behind it
Similarity: 0.0702
